In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import pathlib
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from natsort import natsorted
from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.evaluation.metrics_utils import default_jsd

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from dmpe.related_work.random_walk import random_walk_control_law

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
full_column_width = 18.2
half_column_width = 8.89

def plot_jsd_model_prediction_relation(
    data_path: pathlib.Path,
    model_class: eqx.Module,
    verbose: bool = False,
    expecting_sub_folders: bool = True,
    penalty_function: callable = None,
    recompute_jsd: bool = False,
    consider_actions: bool = True,
):
    means = []
    medians = []
    jsds = []
    colors = []

    color_cycle = plt.rcParams["axes.prop_cycle"]()
    color_mapping = [next(color_cycle)["color"] for _ in range(15)]

    result_paths = (
        glob.glob(str(data_path) + "/**/*.eqx") if expecting_sub_folders else glob.glob(str(data_path) + "/*.eqx")
    )

    n_results = len(result_paths)
    print("# or results:", n_results)
    print(80 * "-")

    for result_path in tqdm(result_paths, total=len(result_paths)):
        result = ModelExpDataResult.from_file(
            filename=result_path,
            model_class=model_class,
        )
        if penalty_function is not None:
            if penalty_function(result.observations, result.actions) > 1:
                continue

        colors.append(result.n_datapoints)

        means.append(jnp.mean(jnp.array(result.model_errors), axis=0)[-1])
        medians.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

        if recompute_jsd:
            jsd_value = default_jsd(
                result.observations,
                result.actions,
                points_per_dim=20,
                bounds=(-1, 1),
                bandwidth=0.08,  # TODO: What about this?!
                target_distribution=None,
                ca=consider_actions,
            )
        else:
            jsd_value = result.data_jsd

        jsds.append(jsd_value)

        if verbose:
            print(result.data_jsd)
            fig, _ = result.visualize()
            plt.show()
            print(80 * "-")

    fig, ax = plt.subplots(1, 1, figsize=(half_column_width, 4.5))

    ax.scatter(
        jsds, medians, s=25, marker="x", c=colors
    )  # , c=next(colors)["color"], label=f"{data_length} data points")

    # ax.set_ylabel("model prediction loss")
    # ax.set_xlabel("JSD")
    ax.set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which="both", alpha=0.3)

    # legend_elements = [Patch(facecolor=color_mapping[idx], label=(idx + 1) * 1000) for idx in range(len(color_mapping))]
    # ax.legend(handles=legend_elements, title=r"\# of datapoints")

    return fig, ax

In [ ]:
# for sys_name in ["fluid_tank", "pendulum", "cart_pole"]:
#     for consider_actions in [True, False]:
#         fig, axs = plot_jsd_model_prediction_relation(
#             data_path=DataPaths().model_learning_cs_out / sys_name / "2step",
#             model_class=NeuralEulerODE,
#             verbose=False,
#             expecting_sub_folders=False,
#             recompute_jsd=True,
#             consider_actions=consider_actions
#         )
#         plt.savefig(f"JSD_LM_{sys_name}_ca_{consider_actions}.pdf", bbox_inches='tight');

## Combined plot

In [ ]:
def plot_jsd_model_prediction_relation_for_multiple_systems(
    data_paths: list[pathlib.Path],
    model_classes: list[eqx.Module],
    consider_actions: bool = True,
):
    fig, axs = plt.subplots(1,3, figsize=(full_column_width, 5))

    for sys_idx, (data_path, model_class) in enumerate(zip(data_paths, model_classes)):
        model_errors = []
        jsds = []
        colors = []

        result_paths = glob.glob(str(data_path) + "/*.eqx")

        n_results = len(result_paths)
        for result_path in tqdm(result_paths, total=len(result_paths)):
            result = ModelExpDataResult.from_file(
                filename=result_path,
                model_class=model_class,
            )
            colors.append(result.n_datapoints)
            model_errors.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

            jsd_value = default_jsd(
                result.observations,
                result.actions,
                points_per_dim=20,
                bounds=(-1, 1),
                bandwidth=0.08,
                target_distribution=None,
                ca=consider_actions,
            )
            jsds.append(jsd_value)

        sc = axs[sys_idx].scatter(
            jsds, model_errors, s=25, marker="x", c=colors
        )
        # plt.colorbar(sc)

    axs[0].set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    for ax, col in zip(
        axs, 
        ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]
    ):
        ax.set_title(col)

    for ax in axs:
        ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.tick_params(which='both', axis="y", direction='in')
        ax.tick_params(which='both', axis="x", direction='in')
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    return fig, axs

In [ ]:
consider_actions = True
fig, axs = plot_jsd_model_prediction_relation_for_multiple_systems(
    data_paths=[
        DataPaths().model_learning_cs_out / sys_name / "2step"
        for sys_name in ["fluid_tank", "pendulum", "cart_pole"]
    ],
    model_classes=[
        NeuralEulerODE,
        NeuralEulerODEPendulum,
        NeuralEulerODECartpole,
    ],
    consider_actions=consider_actions,
)

axs[0].set_yticks([], minor=True)
axs[0].set_xticks([], minor=True)
axs[1].set_yticks([], minor=True)
axs[1].set_xticks([], minor=True)
axs[2].set_yticks([], minor=True)
axs[2].set_xticks([], minor=True)

axs[0].set_xticks([1e-1], minor=False)
axs[0].set_xticks([5e-2, 2e-1], [r"$5 \times 10^{-2}$", r"$2 \times 10^{-1}$"], minor=True)

axs[1].set_xticks([1e-1], minor=False)
axs[1].set_xticks([5e-2, 3e-1], [r"$5 \times 10^{-2}$", r"$3 \times 10^{-1}$"], minor=True)
axs[2].set_xticks([4e-1, 5e-1, 6e-1], minor=False)
# axs[1].set_xticks([5e-2, 3e-1], [r"$5 \times 10^{-2}$", r"$3 \times 10^{-1}$"], minor=True)

fig.savefig(f"JSD_LM_all_systems_ca_{consider_actions}.svg", bbox_inches='tight');
plt.show()

In [ ]:
# colorbar

# Create a colormap and a normalization
cmap = plt.cm.viridis
norm = mpl.colors.Normalize(vmin=1_000, vmax=15_000)

# Create a dummy ScalarMappable to use for the colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Needed for some versions of matplotlib

# Create the figure and add only a colorbar
fig, ax = plt.subplots(figsize=(full_column_width, 1))
fig.subplots_adjust(bottom=0.5)

cbar = fig.colorbar(sm, cax=ax, orientation='horizontal', label=r"\# of data points")

plt.savefig("colorbar_n_datapoints.svg", bbox_inches="tight")
plt.show()

In [ ]:
consider_actions = False
fig, axs = plot_jsd_model_prediction_relation_for_multiple_systems(
    data_paths=[
        DataPaths().model_learning_cs_out / sys_name / "2step"
        for sys_name in ["fluid_tank", "pendulum", "cart_pole"]
    ],
    model_classes=[
        NeuralEulerODE,
        NeuralEulerODEPendulum,
        NeuralEulerODECartpole,
    ],
    consider_actions=consider_actions,
)

axs[0].set_yticks([], minor=True)
axs[0].set_xticks([], minor=True)
axs[1].set_yticks([], minor=True)
axs[1].set_xticks([], minor=True)
axs[2].set_yticks([], minor=True)
axs[2].set_xticks([], minor=True)

axs[1].set_xticks([1e-1], minor=False)
axs[1].set_xticks([3e-2, 3e-1], [r"$3 \times 10^{-2}$", r"$3 \times 10^{-1}$"], minor=True)
axs[2].set_xticks([4e-1, 6e-1], [r"$4 \times 10^{-1}$", r"$6 \times 10^{-1}$"], minor=True)
fig.savefig(f"JSD_LM_all_systems_ca_{consider_actions}.pdf", bbox_inches='tight');

## Only actions in jsd:

In [ ]:
from dmpe.utils.metrics import JSDLoss
from dmpe.utils.density_estimation import (
    DensityEstimate,
    build_grid,
    update_density_estimate_multiple_observations,
)

In [ ]:
def evaluate_datapoints_with_jsd(data_points, points_per_dim, bounds, bandwidth, target_distribution):
    dim = data_points.shape[-1]
    n_grid_points = points_per_dim**dim

    density_estimate = DensityEstimate(
        p=jnp.zeros([n_grid_points, 1]),
        z_g=build_grid(dim, low=bounds[0], high=bounds[1], points_per_dim=points_per_dim),
        bandwidth=jnp.array([bandwidth]),
        n_observations=jnp.array([0]),
    )

    if data_points.shape[0] > 5000:
        # if there are too many datapoints at once, split them up and add
        # them in smaller chunks to the density estimate

        block_size = 500

        for n in range(0, data_points.shape[0] + 1, block_size):
            density_estimate = update_density_estimate_multiple_observations(
                density_estimate,
                data_points[n : min(n + block_size, data_points.shape[0])],
            )
    else:
        density_estimate = update_density_estimate_multiple_observations(
            density_estimate,
            data_points,
        )

    if target_distribution is None:
        target_distribution = jnp.ones(density_estimate.p.shape)
        target_distribution /= jnp.sum(target_distribution)

    return JSDLoss(
        p=density_estimate.p / jnp.sum(density_estimate.p),
        q=target_distribution,
    )

In [ ]:
def plot_jsd_actions_only_model_prediction_relation_for_multiple_systems(
    data_paths: list[pathlib.Path],
    model_classes: list[eqx.Module],
):
    fig, axs = plt.subplots(1,3, figsize=(full_column_width, 5))

    for sys_idx, (data_path, model_class) in enumerate(zip(data_paths, model_classes)):
        model_errors = []
        jsds = []
        colors = []

        result_paths = glob.glob(str(data_path) + "/*.eqx")

        n_results = len(result_paths)
        for result_path in tqdm(result_paths, total=len(result_paths)):
            result = ModelExpDataResult.from_file(
                filename=result_path,
                model_class=model_class,
            )
            colors.append(result.n_datapoints)
            model_errors.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

            jsd_value = evaluate_datapoints_with_jsd(
                result.actions,
                points_per_dim=20,
                bounds=(-1, 1),
                bandwidth=0.08,
                target_distribution=None,
            )
            jsds.append(jsd_value)

        sc = axs[sys_idx].scatter(
            jsds, model_errors, s=25, marker="x", c=colors
        )
        # plt.colorbar(sc)

    axs[0].set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    for ax, col in zip(
        axs, 
        ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]
    ):
        ax.set_title(col)

    for ax in axs:
        ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.tick_params(which='both', axis="y", direction='in')
        ax.tick_params(which='both', axis="x", direction='in')
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    return fig, axs

In [ ]:
fig, axs = plot_jsd_actions_only_model_prediction_relation_for_multiple_systems(
    data_paths=[
        DataPaths().model_learning_cs_out / sys_name / "2step"
        for sys_name in ["fluid_tank", "pendulum", "cart_pole"]
    ],
    model_classes=[
        NeuralEulerODE,
        NeuralEulerODEPendulum,
        NeuralEulerODECartpole,
    ],
)

axs[0].set_yticks([], minor=True)
axs[0].set_xticks([], minor=True)
axs[1].set_yticks([], minor=True)
axs[1].set_xticks([], minor=True)
axs[2].set_yticks([], minor=True)
axs[2].set_xticks([], minor=True)

axs[1].set_xticks([6e-3, 3e-2], [r"$6 \times 10^{-3}$", r"$3 \times 10^{-2}$"], minor=True)
axs[1].set_xticks([], minor=False)

plt.savefig(f"JSD_LM_all_systems_ca_only.pdf", bbox_inches='tight');

## with color coded algos:

In [ ]:
fluid_tank_ids = dict(
    dmpe=[
        '2025-08-27_14-55-49',
        '2025-08-27_14-57-26',
        '2025-08-27_14-59-04',
        '2025-08-27_15-00-41',
        '2025-08-27_15-02-19',
    ],
    pm_dmpe=[
        '2025-08-28_10-31-04',
        '2025-08-28_10-36-03',
        '2025-08-28_10-41-00',
        '2025-08-28_10-45-58',
        '2025-08-28_10-50-54',
    ],
    sgoats=[
        '2025-07-23_17-05-04',
        '2025-07-23_17-25-52',
        '2025-07-23_17-46-10',
        '2025-07-23_18-04-37',
        '2025-07-23_18-25-10',
    ],
    igoats=[
        '2025-07-23_16-44-48',
        '2025-07-23_17-03-43',
        '2025-07-23_17-22-58',
        '2025-07-23_17-40-58',
        '2025-07-23_17-59-50',
    ],
    random_walk=[
        '2025-07-24_11-19-31',
        '2025-07-24_11-19-43',
        '2025-07-24_11-19-54',
        '2025-07-24_11-20-06',
        '2025-07-24_11-20-18',
    ],
)

pendulum_ids = dict(
    dmpe=[
        '2025-08-27_14-57-18',
        '2025-08-27_15-00-42',
        '2025-08-27_15-04-07',
        '2025-08-27_15-07-34',
        '2025-08-27_15-11-00',
    ],
    pm_dmpe=[
        '2025-08-28_10-35-03',
        '2025-08-28_10-44-58',
        '2025-08-28_10-54-58',
        '2025-08-28_11-04-31',
        '2025-08-28_11-14-47',
    ],
    sgoats=[
        '2025-07-23_17-04-21',
        '2025-07-23_17-22-38',
        '2025-07-23_17-40-52',
        '2025-07-23_18-00-08',
        '2025-07-23_18-17-27',
    ],
    igoats=[
        '2025-07-23_16-44-37',
        '2025-07-23_17-02-47',
        '2025-07-23_17-21-38',
        '2025-07-23_17-40-06',
        '2025-07-23_17-59-04',
    ],
    random_walk=[
        '2025-07-24_11-15-20',
        '2025-07-24_11-15-32',
        '2025-07-24_11-15-44',
        '2025-07-24_11-15-55',
        '2025-07-24_11-16-06',
    ],
)

cart_pole_ids = dict(
    dmpe=[
        '2025-08-27_15-13-55',
        '2025-08-27_15-34-19',
        '2025-08-27_15-54-39',
        '2025-08-27_16-15-02',
        '2025-08-27_16-35-27',
    ],
    pm_dmpe=[
        '2025-08-28_10-44-48',
        '2025-08-28_11-04-06',
        '2025-08-28_11-23-21',
        '2025-08-28_11-42-41',
        '2025-08-28_12-02-02',
    ],
    sgoats=[
        '2025-07-24_11-52-44',
        '2025-07-24_12-31-52',
        '2025-07-24_13-10-21',
        '2025-07-24_13-50-30',
        '2025-07-24_14-28-45',
    ],
    igoats=[
        '2025-07-23_17-34-12',
        '2025-07-23_18-35-44',
        '2025-07-23_19-39-24',
        '2025-07-23_20-44-08',
        '2025-07-23_21-51-53',
    ],
    random_walk=[
        '2025-07-24_11-19-00',
        '2025-07-24_11-19-13',
        '2025-07-24_11-19-26',
        '2025-07-24_11-19-39',
        '2025-07-24_11-19-52',
    ],
)

def get_algo(exp_id, ids):
    for algo_name, exp_ids_in_algo in ids.items():
        if exp_id in exp_ids_in_algo:
            return algo_name
    return "fail"

get_fluid_tank_algo = partial(get_algo, ids=fluid_tank_ids)
get_pendulum_algo = partial(get_algo, ids=pendulum_ids)
get_cart_pole_algo = partial(get_algo, ids=cart_pole_ids)

In [ ]:
def plot_jsd_model_prediction_relation_for_multiple_systems_color_coded_algos(
    data_paths: list[pathlib.Path],
    model_classes: list[eqx.Module],
    consider_actions: bool = True,
    get_algo_functions: list = [],
):
    fig, axs = plt.subplots(1,3, figsize=(full_column_width, 5))
    color_cycle = plt.rcParams["axes.prop_cycle"]()
    color_map = dict(
        pm_dmpe=next(color_cycle)["color"],
        dmpe=next(color_cycle)["color"],
        sgoats=next(color_cycle)["color"],
        fail=next(color_cycle)["color"],
        igoats=next(color_cycle)["color"],
        random_walk=next(color_cycle)["color"],
    )

    for sys_idx, (data_path, model_class, get_algo_for_exp) in enumerate(zip(data_paths, model_classes, get_algo_functions)):
        model_errors = []
        jsds = []
        colors = []

        result_paths = glob.glob(str(data_path) + "/*.eqx")

        n_results = len(result_paths)
        for result_path in tqdm(result_paths, total=len(result_paths)):
            result = ModelExpDataResult.from_file(
                filename=result_path,
                model_class=model_class,
            )
            algo_name = get_algo_for_exp(result.exp_id)
            
            
            colors.append(color_map[algo_name])
            model_errors.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

            jsd_value = default_jsd(
                result.observations,
                result.actions,
                points_per_dim=20,
                bounds=(-1, 1),
                bandwidth=0.08,
                target_distribution=None,
                ca=consider_actions,
            )
            jsds.append(jsd_value)

        sc = axs[sys_idx].scatter(
            jsds, model_errors, s=25, marker="x", c=colors
        )
        # plt.colorbar(sc)

    axs[0].set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    for ax, col in zip(
        axs, 
        ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]
    ):
        ax.set_title(col)

    for ax in axs:
        ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    return fig, ax

In [ ]:
for consider_actions in [True, False]:
    fig, axs = plot_jsd_model_prediction_relation_for_multiple_systems_color_coded_algos(
        data_paths=[
            DataPaths().model_learning_cs_out / sys_name / "2step"
            for sys_name in ["fluid_tank", "pendulum", "cart_pole"]
        ],
        model_classes=[
            NeuralEulerODE,
            NeuralEulerODEPendulum,
            NeuralEulerODECartpole,
        ],
        get_algo_functions=[
            get_fluid_tank_algo,
            get_pendulum_algo,
            get_cart_pole_algo,
        ],
        consider_actions=consider_actions,
    )
    plt.show()

In [ ]:
color_cycle = plt.rcParams["axes.prop_cycle"]()
color_map = dict(
    pm_dmpe=next(color_cycle)["color"],
    dmpe=next(color_cycle)["color"],
    sgoats=next(color_cycle)["color"],
    fail=next(color_cycle)["color"],
    igoats=next(color_cycle)["color"],
    random_walk=next(color_cycle)["color"],
)

fig, axs = plt.subplots(1,1)
for algo, color in color_map.items():
    axs.plot(np.linspace(-1,1,10), np.linspace(-1,1,10)**2, c=color, label=algo)

plt.legend()
plt.show()